https://github.com/KeremOzemre/Computational-Social-Science.git

### Part 1

#### 1:
Centola’s study uses custom-made data because the researcher designed an online experiment with controlled network structures. The main advantage was that researchers could control the study conditions, which makes it easier to identify what causes what, reduce the influence of other factors, and understand how social influence (like reinforcement from others) affects behaviour. This fits the strengths of experiments: like high internal validity and clear controlled conditions. However, custom-made data are costly, involve smaller samples, and may create artificial settings that reduce real-world realism.

Nicolaides’ study uses ready-made data, analysing large-scale fitness-tracking and social network traces. Advantages include huge samples, real-world behaviour, and long time scales, which makes the results more representative of real-world behaviour and allows researchers to detect patterns more reliably because of the large amount of data. But, as discussed in Chapter 2.3, ready-made data can be incomplete, biased, dirty, and affected by confounding factors, making causal claims harder.

#### 2:
The differences between custom-made and ready-made data influence how the results should be interpreted. In Centola’s experiment the controlled design makes it easier to interpret the results as causal, meaning that differences in behaviour can more confidently be explained by the network structure itself. However, because the setting was artificial and more limited, the findings might not fully show how behavior spreads in real-world social networks. In contrast, Nicolaides’ study uses real-world data from a large population, which makes the results more realistic and broadly aplicable. However, because the data were not created for research purposes, there may be hidden biases and confounding factors meaning the results should be interpreted more as strong associations rather than clear causal effects. Overall for the interpretation, Centola provides stronger causal evidence but lower realism, while Nicolaides offers higher realism and scale but more uncertainty about causality.



### Part 2

In [ ]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
from itertools import islice
from joblib import Parallel, delayed
from tqdm import tqdm
API_KEY = os.getenv("API_KEY")
authors_file = open('authors.txt','r')

### Extracting Open Alex by calling the API on names extracted from Web Scraping

In [ ]:
import requests
import pandas as pd
import time

rows = []
batch_size = 25 # number of authors per batch
sleep_time = 1   # seconds to wait between batches

authors = [a.strip() for a in authors_file if a.strip()]
total = len(authors)

for start in range(0, total, batch_size):
    batch = authors[start:start + batch_size]
    for author in batch:
        URL = "https://api.openalex.org/authors"
        params = {
            "search": author,
            "select": "display_name,works_count,id,works_api_url,last_known_institutions,summary_stats",
            "api_key": API_KEY
        }
        try:
            response = requests.get(URL, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed for {author}: {e}")
            continue

        if data.get("results"):
            author_data = data["results"][0]
            h_index = author_data.get("summary_stats", {}).get("h_index")
            institutions = author_data.get("last_known_institutions", [])
            country_code = institutions[0].get("country_code") if institutions else None

            rows.append({
                "searched_name": author,
                "display_name": author_data.get("display_name"),
                "works_count": author_data.get("works_count"),
                "author_id": author_data.get("id"),
                "works_api_url": author_data.get("works_api_url"),
                "h_index": h_index,
                "country_code": country_code
            })
        else:
            print(f"No results found for author: {author}")

    print(f"Processed batch {start // batch_size + 1} / {total // batch_size + 1}")
    time.sleep(sleep_time)  # pause between batches to avoid throttling

df = pd.DataFrame(rows)
df.to_csv("IC2S2_authors.csv", index=False)

### Part 3

### Filtering Authors by work count

In [ ]:
df1 = pd.read_csv("authors_data.csv")

indexes = df1.index[
    (df1["works_count"] >= 5) &
    (df1["works_count"] <= 5000)
].tolist()

filtered_author_list = df1.loc[indexes, "author_id"]

### Calling API for all works of all IC2S2 authors

In [ ]:
# Concept filter ensuring only works within the intersection of computational social science and quantitative disciplines
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
quant_concepts = ["C33923547", "C121332964", "C41008148"]

concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

BASE_URL = "https://api.openalex.org/works"

# send requests in chunks (25 max permitted by open alex) to filter multiple authors in the same request for faster extraction
def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

# Created function, instead of for loop as we can call the same function in parallel to again fasten the process
def process_single_batch(author_batch, batch_num):
    
    rows2_local = []
    rows3_local = []
    
    author_filter = "author.id:" + "|".join(author_batch)

    filter_string = (
        f"{author_filter},"
        f"cited_by_count:>10,"
        f"authors_count:<10,"
        f"{concept_filter1},"
        f"{concept_filter2}"
    )
    
    cursor = "*"
    
    while cursor:
        params = {
            "filter": filter_string,
            "select": "id,publication_year,cited_by_count,title,abstract_inverted_index,authorships",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }
        
        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Error in batch {batch_num}: {e}")
            break
        
        results = data.get("results", [])
        if not results:
            break
        
        for item in results:
            work_id = item.get("id")
            
            # find author id
            author_ids = [
                auth["author"]["id"]
                for auth in item.get("authorships", [])
                if auth.get("author") and auth["author"].get("id")
            ]
        
            
            rows2_local.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "author_ids": author_ids,
            })
            
            rows3_local.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index"),
            })
        
        cursor = data["meta"].get("next_cursor")
        sleep(0.05)  # Small delay between pages to avoid going over the rate limit 
    
    return rows2_local, rows3_local

# Create batches
author_batches = list(chunked(filtered_author_list, 25))

# Process batches in parallel with tqdm progress bar
results = Parallel(n_jobs=4, verbose=0)(
    delayed(process_single_batch)(batch, i+1) 
    for i, batch in enumerate(tqdm(author_batches, desc="Processing batches"))
)

# Combine results
rows2 = []
rows3 = []
for batch_rows2, batch_rows3 in results:
    rows2.extend(batch_rows2)
    rows3.extend(batch_rows3)

# Create DataFrames
df2 = pd.DataFrame(rows2)
df3 = pd.DataFrame(rows3)

print(f"\nComplete!")
print(f"Works retrieved: {len(df2)}")
print(f"Abstracts retrieved: {len(df3)}")

In [ ]:
df2.to_csv("IC2S2_papers.csv", index=False)
df3.to_csv("IC2S2_abstracts.csv", index=False)